In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math
import scipy.stats as stats

In [3]:
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.neighbors import NearestNeighbors, KNeighborsClassifier, KNeighborsRegressor
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, mean_squared_error, r2_score

In [10]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.6/386.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 15.4 MB/s eta 0:00:00


In [5]:
df1= pd.read_csv("../filtered_data_v1.3.csv")

Asthma Target

In [ ]:
from sklearn.model_selection import train_test_split

# Clean the noisy data
df_asthma= df1.copy()
df_asthma= df_asthma[df_asthma['MCQ010'].isin([1.0, 2.0])]  # Keep only 1.0 and 2.0

# Split features and target
X = df_asthma.drop(columns=['MCQ010'])
y = df_asthma['MCQ010']

# Train/test/validation split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=35, stratify=y)
X_test, X_val, y_test, y_val = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=35)

In [ ]:
import optuna
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import cross_val_score
from imblearn.pipeline import Pipeline as ImbPipeline

def objective(trial):
    k = trial.suggest_categorical('n_neighbors', [3, 5, 7, 9])

    pipeline = ImbPipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier(n_neighbors=k))
    ])

    # Cross-validation on training set
    score = cross_val_score(pipeline, X_train, y_train, scoring='f1', cv=3, n_jobs=-1)
    return score.mean()

# Optuna study
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100, show_progress_bar=True)

In [12]:
# Rebuild final model with best parameters
best_k = study.best_params['n_neighbors']
final_pipeline = ImbPipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=best_k))
])

final_pipeline.fit(X_train, y_train)

# Evaluate on validation set
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

val_probs = final_pipeline.predict_proba(X_val)[:, 1]
val_preds = final_pipeline.predict(X_val)

print(classification_report(y_val, val_preds))
print("ROC-AUC:", roc_auc_score(y_val, val_probs))
print("PR-AUC:", average_precision_score(y_val, val_probs))

NameError: name 'study' is not defined

CVD_Target

In [ ]:
from sklearn.model_selection import train_test_split

# Clean the noisy data
df_CVD= df1.copy()

# Filter valid binary values
df_CVD['CVD_combined'] = df_CVD['CVD_combined'].apply(lambda x: 1 if x > 1 else x)

X = df_CVD.drop(columns=['CVD_combined', 'MCQ160C', 'MCQ160B', 'MCQ160E'])
y = df_CVD['CVD_combined']

# Train/test/validation split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=45, stratify=y)
X_test, X_val, y_test, y_val = train_test_split(X_temp, y_temp, test_size=0.5, random_state=45, stratify=y_temp)

In [ ]:
import optuna
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import cross_val_score
from imblearn.pipeline import Pipeline as ImbPipeline

def objective(trial):
    k = trial.suggest_categorical('n_neighbors', [3, 5, 7, 9])

    pipeline = ImbPipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier(n_neighbors=k))
    ])

    # Cross-validation on training set
    score = cross_val_score(pipeline, X_train, y_train, scoring='f1', cv=3, n_jobs=-1)
    return score.mean()

# Optuna study
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100, show_progress_bar=True)

In [ ]:
# Rebuild final model with best parameters
best_k = study.best_params['n_neighbors']
final_pipeline = ImbPipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=best_k))
])

final_pipeline.fit(X_train, y_train)

# Evaluate on validation set
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

val_probs = final_pipeline.predict_proba(X_val)[:, 1]
val_preds = final_pipeline.predict(X_val)

print(classification_report(y_val, val_preds))
print("ROC-AUC:", roc_auc_score(y_val, val_probs))
print("PR-AUC:", average_precision_score(y_val, val_probs))